# Заняття 10 · LIVE DEMO 1
## Черги на дефолтах проти Latest(1)

Справжній GStreamer на одному комп’ютері: відправник `appsrc → x264enc → RTP/UDP`, приймач `udpsrc → rtpjitterbuffer → avdec_h264 → appsink`.
Кожен кадр несе штрихкод зі своїм номером (дві копії з контрольною сумою), тому **вік кадру = час прийому − час відправлення того самого кадру** за одним годинником — вимірювання в один бік тут чесне.

| Крок | Що робимо |
|---|---|
| §1 | Відправник і приймач без навантаження: базова затримка |
| §2 | appsink без ліміту + «детектор» на 70 мс — вік кадру росте сам |
| §3 | Latest(1): appsink на один кадр — вік сталий, рахуємо дропи |
| §4 | Трекер: dt = 1 оновлення проти dt з PTS на кадрах, які реально дійшли до детектора |

Мітка на слайдах: «LIVE DEMO 1 · lesson10_demo1_queues_latest.ipynb» (слайд 17, розділ 02 «GStreamer як граф»); типові значення черг — слайд 12, лаг росте без помилок — слайд 13, Latest(1) — слайд 14, трекер на нерівному кроці — слайд 15, time-to-track — слайд 16. GStreamer живе в `streamlab.py` / `gstworker.py`, стиль графіків — у `panels.py`.

In [ ]:
# streamlab запускає GStreamer в окремому процесі (gstworker.py), тож у ноутбуці gi не імпортується
import numpy as np
import matplotlib.pyplot as plt
import streamlab as sl
from panels import PAL, set_style

set_style()
print(sl.gst_version(), '| appsink leaky-type (1.28+):', sl.appsink_has_leaky_type())

sc = sl.Scene(640, 360, 30)
ENC = 'tune=zerolatency speed-preset=superfast bitrate=1000 key-int-max=30'
T_INF = 0.070            # «детектор»: 70 мс на кадр
SECONDS = 10

### 1. Відправник і приймач без навантаження
Кадр 640×360 з текстурою, рухомою ціллю (червоний прямокутник) і штрихкодом номера зверху та знизу. Спершу — без детектора: скільки коштує сам ланцюг «кодування → RTP → декодування» на цьому комп’ютері.

Анімація — кадри, які **декодував приймач** (кожен другий, зменшені до 480 px); праворуч синхронно будується графік віку кадру, у підписі — номер кадру зі штрихкоду і його вік.

In [ ]:
from panels import frames_player

rec0, tx0, _ = sl.run_stream(sc, 3, ENC, keep_every=1)          # keep_every=1 — приймач віддає й самі декодовані кадри
ok0 = [r for r in rec0 if r['idx'] in tx0.t_send]
age0 = np.array([r['age'] for r in ok0])
print(f'кадрів: {len(age0)}  вік p50 = {np.median(age0):.1f} мс  p99 = {np.percentile(age0, 99):.1f} мс')

shown = ok0[::2]
frames_player([r['img'] for r in shown], [f"кадр {r['idx']:3d} · вік {r['age']:5.1f} мс" for r in shown], fps=15,
              chart=dict(x=[r['idx'] for r in ok0], y=age0, at=[r['idx'] for r in shown],
                         title='Вік кадру без детектора', xlabel='кадр', ylabel='мс'))

### 2. Дефолти: appsink без ліміту і повільний детектор
`appsink max-buffers=0 drop=false` — так appsink налаштований за замовчуванням: черга без меж. Детектор забирає кадр, «думає» 70 мс, бере наступний.

**Розрахунок до запуску.** Обробляємо 1 / 0.070 ≈ 14.3 кадру/с, приходить 30. Кожен оброблений кадр старший за попередній на 70 − 33.3 = 36.7 мс, тобто вік росте на **1 − T / T_inf ≈ 0.52 с щосекунди**.

In [ ]:
rec_fifo, tx_fifo, _ = sl.run_stream(sc, SECONDS, ENC, max_buffers=0, drop=False, work_s=T_INF, settle_s=0.3)


def series(rec, tx):
    """Час прийому від першого кадру (с), вік кадру (мс) і номер — лише для кадрів із прочитаним штрихкодом."""
    r = [x for x in rec if x['idx'] in tx.t_send]
    t = np.array([x['t_rx'] for x in r])
    return t - t[0], np.array([x['age'] for x in r]), np.array([x['idx'] for x in r])


t_f, a_f, i_f = series(rec_fifo, tx_fifo)
k = len(t_f) // 4
slope = np.polyfit(t_f[k:], a_f[k:] / 1000, 1)[0]
model = 1 - (1 / 30) / T_INF
print(f'нахил віку: виміряно {slope:.3f} с/с, модель {model:.3f} с/с; останній кадр старший за {a_f[-1] / 1000:.2f} с')

### 3. Latest(1): поштова скринька на один кадр
`appsink max-buffers=1 drop=true` (з GStreamer 1.28 — `leaky-type=downstream`, `drop` позначено застарілою). Детектор завжди отримує найсвіжіший кадр; старі викидаються.

In [ ]:
rec_lat, tx_lat, _ = sl.run_stream(sc, SECONDS, ENC, max_buffers=1, drop=True, work_s=T_INF, settle_s=0.3)
t_l, a_l, i_l = series(rec_lat, tx_lat)
drops = int(np.sum(np.diff(i_l) - 1))
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(t_f, a_f / 1000, color=PAL['re'], label=f'без ліміту: +{slope:.2f} с/с')
ax.plot(t_l, a_l / 1000, color=PAL['te'], label=f'Latest(1): p50 {np.median(a_l):.0f} мс')
ax.set_xlabel('час, с'); ax.set_ylabel('вік кадру, с'); ax.set_title('Вік кадру, який бере детектор'); ax.legend()
plt.tight_layout(); plt.show()
print(f'Latest(1): вік p50 = {np.median(a_l):.0f} мс, p99 = {np.percentile(a_l, 99):.0f} мс; дропнуто {drops} кадрів із {i_l[-1] + 1}')

### 4. Трекер на кадрах, які реально дійшли до детектора
Latest(1) робить крок між детекціями нерівним: дивимося на справжні номери кадрів із §3. Імітуємо детектор (істинна позиція цілі + шум 2 px) і порівнюємо два фільтри Калмана з постійною швидкістю:

* **dt з PTS** — крок у секундах між кадрами, які бачив детектор;
* **dt = 1** — крок = «одне оновлення», як у трекерах, налаштованих у кадрах.

Шум процесу однаковий фізично (те саме прискорення в px/с²), тому різниця — лише в тому, як враховано час. Додатково — стрибки до 6–9 кадрів (зупинки детектора), які на завантаженому борту трапляються.

In [ ]:
def tracker_errors(idx, mode, q_acc=2.5e5, seed=0):
    """Помилка прогнозу Калмана (px) на послідовності кадрів idx; mode='pts' — dt у секундах, 'step' — dt = 1."""
    rng = np.random.default_rng(seed)
    T_nom = np.mean(np.diff(idx)) / 30
    kf = sl.CVKalman(q=q_acc if mode == 'pts' else q_acc * T_nom ** 4, dt_mode=mode)
    errs = []
    for k, i in enumerate(idx):
        z, t = sc.target(i) + rng.normal(0, 2, 2), i / 30
        if kf.x is not None and k > 5:
            dt = (t - kf.t) if mode == 'pts' else 1.0
            errs.append(np.linalg.norm(kf.predict_at(dt) - sc.target(i)))
        kf.update(z, t)
    return np.array(errs)


rng = np.random.default_rng(1)
stalled = [0]
for g in np.diff(i_l):                          # ті самі кроки + 10 % зупинок на 6–9 кадрів
    stalled.append(stalled[-1] + (int(g) if rng.random() > 0.1 else int(rng.integers(6, 10))))
GATE = 15
rows = []
for name, idx in (('кадри з §3', i_l), ('§3 + зупинки детектора', np.array(stalled))):
    for mode in ('pts', 'step'):
        e = tracker_errors(idx, mode)
        rows.append((name, mode, np.median(e), np.percentile(e, 95), (e > GATE).mean()))
print(f'{"послідовність":26s} {"dt":5s} {"p50, px":>8s} {"p95, px":>8s} {"> гейта":>8s}')
for r in rows:
    print(f'{r[0]:26s} {r[1]:5s} {r[2]:8.1f} {r[3]:8.1f} {r[4]:8.1%}')
upd = 1 / np.mean(np.diff(t_l))
print(f'\nдетектор оновлює трекер з частотою {upd:.1f} Гц: track_buffer=30 «оновлень» = {30 / upd:.1f} с, а не 1 с')

### Висновки для слайда
* Без обмеження черги вік кадру росте лінійно, і нахил збігається з моделлю `1 − T / T_inf`: «повільний детектор» перетворюється на «відео з минулого».
* Latest(1) тримає вік на рівні інференсу плюс частини періоду кадру; ціна — дропи, тобто нерівний крок для трекера.
* Поки крок майже рівний, dt з PTS дає невеликий виграш. Коли детектор зупиняється, фільтр із кроком «одне оновлення» помиляється в прогнозі в кілька разів частіше, ніж дозволяє гейт асоціації: крок і вік треку беремо з PTS.
* На однопроцесорній машині частина затримки — конкуренція за CPU; на своєму залізі повторіть §1–§3 і порівняйте.